# TTS API บน Google Colab (VoxCPM2 / OmniVoice)

1. Runtime → Change runtime type → **T4 GPU**
2. เลือก `ENGINE` ในช่อง 1 (ค่าเริ่ม `voxcpm` = VoxCPM2, weights Apache-2.0 ใช้เชิงพาณิชย์ได้ · `omnivoice` = weights CC-BY-NC ห้ามใช้เชิงพาณิชย์)
3. กด Runtime → Run all
4. รอช่อง 3 พิมพ์ `URL` กับ `KEY` → เอาไปใส่ `client.py` / Dub Studio V4 ที่เครื่องเรา

ปิดแท็บ/หลุด runtime = URL ตาย ต้อง Run all ใหม่ (URL ใหม่ทุกรอบ)

In [ ]:
# 1) ดึงโค้ดจาก GitHub + ติดตั้งโมเดล
ENGINE = "voxcpm"   # "voxcpm" (VoxCPM2) หรือ "omnivoice"
REPO = "https://github.com/zannbls-bot/omnivoice-colab"
!nvidia-smi --query-gpu=name,memory.total --format=csv
!rm -rf /content/ovapi && git clone -q --depth 1 $REPO /content/ovapi
!pip install -q $ENGINE   # package name = engine name

In [ ]:
# 1b) งานหนักของ Dub Studio: แยกเสียงพูด (BS-Roformer) · แยกเอฟเฟกต์ (Bandit-Plus DnR แบบโปรเจค 3-3) · ถอดซับ (faster-whisper large-v3)
!pip install -q "audio-separator[gpu]" faster-whisper ml_collections omegaconf spafe asteroid pytorch_lightning torchmetrics pedalboard loralib
![ -d /content/MSST ] || git clone -q --depth 1 https://github.com/ZFTurbo/Music-Source-Separation-Training /content/MSST
![ -s /content/MSST/model_bandit_plus_dnr_sdr_11.47.chpt ] || wget -q -O /content/MSST/model_bandit_plus_dnr_sdr_11.47.chpt https://github.com/ZFTurbo/Music-Source-Separation-Training/releases/download/v.1.0.3/model_bandit_plus_dnr_sdr_11.47.chpt
!ls -la /content/MSST/model_bandit_plus_dnr_sdr_11.47.chpt

In [ ]:
# 2) เปิดเซิร์ฟเวอร์ (ครั้งแรกโหลดโมเดลหลายนาที)
import os, secrets, subprocess, time, urllib.request
subprocess.run(["pkill", "-f", "ovapi/server"]); time.sleep(3)  # rerun = stop old server first
KEY = secrets.token_urlsafe(16)
env = dict(os.environ, OV_KEY=KEY, OV_PORT="8000")
SERVER = {"voxcpm": "server_voxcpm.py", "omnivoice": "server.py"}[ENGINE]
srv = subprocess.Popen(["python", "/content/ovapi/" + SERVER], env=env,
                       stdout=open("/content/server.log", "w"), stderr=subprocess.STDOUT)
for _ in range(180):
    try:
        print(urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=5).read().decode()); break
    except Exception:
        if srv.poll() is not None:
            print(open("/content/server.log").read()[-3000:]); raise SystemExit("server died")
        time.sleep(5)
else:
    print(open("/content/server.log").read()[-3000:]); raise SystemExit("server not ready in 15 min")

In [ ]:
# 3) เปิดอุโมงค์ Cloudflare (ฟรี ไม่ต้องสมัคร) → ได้ URL สาธารณะ
import re
subprocess.run(["pkill", "-f", "cloudflared tunnel"])
![ -x /content/cloudflared ] || (wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared)
tun = subprocess.Popen(["/content/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8000"],
                       stdout=open("/content/tunnel.log", "w"), stderr=subprocess.STDOUT)
URL = None
for _ in range(60):
    m = re.search(r"https://(?!api\.)[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if m: URL = m.group(0); break
    time.sleep(2)
print("URL =", URL)
print("KEY =", KEY)
print()
print(f'python client.py --url {URL} --key {KEY} --language th --text "สวัสดีครับ ทดสอบเสียง" --out test.wav')

In [ ]:
# 4) (ไม่บังคับ) ทดสอบในนี้เลย
from IPython.display import Audio
import json
req = urllib.request.Request("http://127.0.0.1:8000/tts",
    json.dumps({"text": "สวัสดีครับ ทดสอบเสียงจาก Colab", "language": "th"}).encode(),
    {"Content-Type": "application/json", "X-API-Key": KEY})
open("/content/t.wav", "wb").write(urllib.request.urlopen(req).read())
Audio("/content/t.wav")

In [ ]:
# 5) ดู log เซิร์ฟเวอร์ (เวลามีปัญหา)
print(open("/content/server.log").read()[-3000:])